In [1]:
from pathlib import Path
import os
from pprint import pprint

from datasets import load_from_disk

import weaviate
from weaviate.classes.config import Configure

DATA_DIR = Path(os.getcwd()).resolve().parent / "data"
DS_PATH = DATA_DIR/ "parsed-recipes"

In [2]:
ds = load_from_disk(DS_PATH)

In [3]:
len(ds)

522517

In [5]:
from collections import Counter
desc_count = Counter(ds["Description"])

In [6]:
desc_count.most_common(20)

[('Make and share this Banana Bread recipe from Food.com.', 96),
 ('Make and share this Beef Stroganoff recipe from Food.com.', 71),
 ('Make and share this Chocolate Chip Cookies recipe from Food.com.', 59),
 ('Make and share this Apple Crisp recipe from Food.com.', 57),
 ('Make and share this Taco Soup recipe from Food.com.', 56),
 ('Make and share this Guacamole recipe from Food.com.', 54),
 ('Make and share this Sweet Potato Casserole recipe from Food.com.', 51),
 ('Make and share this Zucchini Bread recipe from Food.com.', 49),
 ('Make and share this Peanut Butter Cookies recipe from Food.com.', 49),
 ('On one of my many cookbook /recipe searches , I found a cookbook called Americana Cookery. I liked the name , but I loved some of the recipes in it even more. Thought I might place some of them here for safe keeping and to share',
  49),
 ('Make and share this Chicken Enchiladas recipe from Food.com.', 48),
 ('Make and share this Broccoli Salad recipe from Food.com.', 47),
 ('Make a

In [7]:
sum(desc_count.values()) / len(ds) 

1.0

In [8]:
import copy
og_count = copy.deepcopy(dict(desc_count))

In [9]:
from tqdm import tqdm

to_remove = []
for desc in tqdm(desc_count.keys()):
    if "food.com" in str(desc).lower():
        to_remove.append(desc)

for d in to_remove:     
    desc_count.pop(d)

print("Percentage with real descriptions:")        
sum(desc_count.values()) / len(ds) 

100%|██████████| 492839/492839 [00:00<00:00, 2682257.47it/s]

Percentage with real descriptions:


0.6253538162394716

In [10]:
unique = [d for d in desc_count if desc_count[d] == 1]
generic = [d for d in og_count if "food.com" in str(d).lower()]
gen =     [d for d in og_count.items() if "food.com" not in str(d[0]).lower() and og_count[d[0]] > 1]

print(f"only appears once:                {len(unique):,}")
print(f"has food.com (meaningless desc):  {len(generic):,}")
print(f"generic but doesn't contain food.com: {len(gen)}")

only appears once:                324,297
has food.com (meaningless desc):  167,685
generic but doesn't contain food.com: 857


In [11]:
max(v for _, v in gen)

49

In [12]:
print("Percentage with unique descriptions:")
len(unique) / len(ds)

Percentage with unique descriptions:


0.6206439216331717

In [ ]:
vec_columns = [
    "name", 
    "category", 
    "keywords", 
    "ingredients", 
    "description"
]

In [14]:
small = ds.select(range(1_000))

In [15]:
!docker compose up -d

/home/zelluzy/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=46262) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[+] up 2/2
 ✔ Container recipe-genome-t2v-transformers-1 Running                       0.0s
 ✔ Container recipe-genome-weaviate-1         Running                       0.0s


In [16]:
import time
time.sleep(10) # in case container isn't up right away

In [17]:
client = weaviate.connect_to_local()

In [18]:
vector_config = Configure.Vectors.text2vec_transformers(
    name="vector",
    source_properties=vec_columns,
    vectorize_collection_name=False,
)

In [23]:
from weaviate.classes.config import Property, DataType

if client.collections.exists("test_recipes"):
    client.collections.delete("test_recipes")

In [25]:
test_recipes = client.collections.create(
    name="test_recipes",
    vector_config=vector_config,
    properties=[
        Property(name='recipe_id',             data_type=DataType.INT),
        Property(name="name",           data_type=DataType.TEXT),
        Property(name="category",       data_type=DataType.TEXT),
        Property(name="keywords",       data_type=DataType.TEXT_ARRAY),
        Property(name="ingredients",    data_type=DataType.TEXT_ARRAY),
        Property(name="description",    data_type=DataType.TEXT),
        # stored, not vectorized - filterable
        Property(name="rating",         data_type=DataType.NUMBER),
        Property(name="review_count",   data_type=DataType.NUMBER),
        Property(name="calories",       data_type=DataType.NUMBER),
        Property(name="protein",        data_type=DataType.NUMBER),
        Property(name="fat",            data_type=DataType.NUMBER),
        Property(name="carbs",          data_type=DataType.NUMBER),
        Property(name="servings",       data_type=DataType.NUMBER),
    ]
)

In [26]:
from typing import Any

def to_props(recipe: dict[str, Any]) -> dict[str, Any]:
    desc = recipe["Description"] or "" 
    filtered_desc = "" if "food.com" in desc.lower() else desc
    return {
        "recipe_id": recipe["RecipeId"],
        "name": recipe["Name"],
        "category": recipe["RecipeCategory"] or "",
        "keywords": recipe["Keywords"],
        "ingredients": recipe["RecipeIngredientParts"],
        "description": filtered_desc,
        "rating": recipe["AggregatedRating"],
        "review_count": recipe["ReviewCount"],
        "calories": recipe["Calories"],
        "protein": recipe["ProteinContent"],
        "fat": recipe["FatContent"],
        "carbs": recipe["CarbohydrateContent"],
        "servings": recipe["RecipeServings"],
    } 

In [27]:
from weaviate.util import generate_uuid5
from tqdm import tqdm

with test_recipes.batch.dynamic() as batch:
    for recipe in tqdm(small, total=len(small)):
        batch.add_object(
            properties=to_props(recipe), # type: ignore
            uuid=generate_uuid5(recipe["RecipeId"]) # type: ignore
        ) 

print(f"failed: {len(test_recipes.batch.failed_objects)}")
test_recipes.batch.failed_objects[:3]

100%|██████████| 1000/1000 [00:02<00:00, 428.53it/s]


failed: 0


[]

In [28]:
small[0]

{'RecipeId': 38,
 'Name': 'Low-Fat Berry Blue Frozen Dessert',
 'AuthorId': 1533,
 'AuthorName': 'Dancer',
 'CookTime': 'PT24H',
 'PrepTime': 'PT45M',
 'TotalTime': 'PT24H45M',
 'DatePublished': '1999-08-09T21:46:00Z',
 'Description': 'Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.',
 'Images': ['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg',
  'https://img.sndimg.com/food/image/upload/w_555,h_4

In [ ]:
from weaviate.classes.query import Filter
print(
    "Total vectorized: "
    f"{test_recipes.aggregate.over_all(total_count=True).total_count}"
)
query = "dessert"
print(
    f"semantic search result for {query}"
    f"{test_recipes.query.near_text(query=query, limit=5)}"
)

with_md_filter = test_recipes.query.near_text(
    query=query,
    filters=Filter.by_property("calories").less_than(500),
    limit=5,
)
pprint(f"calorie filter: {with_md_filter.objects[0]}")

Total vectorized: 1000
semantic search result for dessertQueryReturn(objects=[Object(uuid=_WeaviateUUIDInt('62d770e4-1e11-5556-8a43-d5fec06b97fa'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'recipe_id': 54, 'protein': 5.0, 'servings': 12.0, 'review_count': 17.0, 'carbs': 67.0, 'name': 'Carrot Cake', 'keywords': ['Vegetable', 'Weeknight', 'Oven', '< 4 Hours'], 'rating': 5.0, 'fat': 27.1, 'description': 'This is one of the few recipes my husband every requested from a coworker and brought home for me to make.', 'category': 'Dessert', 'ingredients': ['carrots', 'eggs', 'white sugar', 'all-purpose flour', 'baking powder', 'baking soda', 'cinnamon', 'salt', 'nutmeg', 'golden raisin', "confectioners' sugar", 'cream cheese', 'light corn syrup', 'vanilla extract'], 'calories': 522.6}, references=None, vector={}, collection='Test_recipes'), Object(uuid=_Wea

In [33]:
client.close()
!docker compose down

/home/zelluzy/.local/share/uv/python/cpython-3.14.3-linux-x86_64-gnu/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=46262) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[+] down 0/2
 ⠋ Container recipe-genome-t2v-transformers-1 Stopping                      0.1s
 ⠋ Container recipe-genome-weaviate-1         Stopping                      0.1s
[+] down 0/2
 ⠙ Container recipe-genome-t2v-transformers-1 Stopping                      0.2s
 ⠙ Container recipe-genome-weaviate-1         Stopping                      0.2s
[+] down 0/2
 ⠹ Container recipe-genome-t2v-transformers-1 Stopping                      0.3s
 ⠹ Container recipe-genome-weaviate-1         Stopping                      0.3s
[+] down 0/2
 ⠸ Container recipe-genome-t2v-transformers-1 Stopping                      0.4s
 ⠸ Container recipe-genome-weaviate-1         Stopping                      0.4s
[+] down 0/2
 ⠼ Container recipe-genome-t2v-transformers-1 Stopping                      0.5s
 ⠼ Container recipe-genome-weaviate-1         Stopping                      0.5s
[+] down 0/2
 ⠴ Container recipe-genome-t2v-transformers-1 Stopping                      0.6s
 ⠴ Container recipe-genome-weav